# Task

With AI-modelling and Data Science there is plenty of opportunity to improve processes or suggest improved ways of doing things. When doing so it is often very smart and efficient (time is a scarce resource) to create a POC (Proof of Concept) which basically is a small demo checking wether it is worthwile going further with something. It is also something concrete which facilitates discussions, do not underestimate the power of that. 

In this example, you are working in a company that sells houses and they have a "manual" process of setting prices by humans. You as a Data Scientist can make this process better by using Machine Learning. Your task is to create a POC that you will present to your team colleagues and use as a source of discussion of wether or not you should continue with more detailed modelling. 

Two quotes to facilitate your reflection on the value of creating a PoC: 

"*Premature optimization is the root of all evil*". 

"*Fail fast*".


**More specifially, do the following:**
1. A short EDA (Exploratory Data Analysis) of the housing data set.
2. Drop the column `ocean_proximity`, then you only have numeric columns which will simplify your analysis. Remember, this is a POC! 
3. You have missing values in your data (not sure you do but you can assume so). Handle this with `SimpleImputer(strategy="median")`. (Check the fantastic Scikit-learn documentation for details.) Notice, the `SimpleImputer` should only be used for transformation on the validation and test data, not fitting.
4. Split your data into `X` and `y`, and then into train, validation and test sets. 
5. Create one `LinearRegression` model and one `Lasso` model. For the `Lasso` model, use `GridSearchCV` to optimize $\alpha$ values. Choose yourself which $\alpha$ values to evaluate.
Use RMSE as a metric to decide which model to choose. 

6. Which model is best on the validation data? 

7. Evaluate your chosen model on the test set using the root mean squared error (RMSE) as the metric. 
What are your conclusions? Note: to be 100% sure, you should re-fit your chosen model on the combination of train+val data. 

8. Do a short presentation (~ 2-5 min) on your POC that you present to your colleagues (no need to prepare anything particular, just talk from the code). Think of:
- What do you want to highlight/present?
- What is your conclusion?
- What could be the next step? Is the POC convincing enough or is it not worthwile continuing? Do we need to dig deeper into this before taking some decisions?

------------
Bonus question for those who have time and are ambitious: Redo everything above (copy your code) but in step 2, include the column `ocean_proximity` which is a categorical column. 

# Code

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


In [21]:
# Below, set your own path where you have stored the data file if it is not in the /data folder. 
housing = pd.read_csv(r'data/housing.csv')
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## EDA

In [22]:
# Check the data.

print("==== INFO ====")
housing.info()

print("\n==== MISSING VALUES ====")
print(housing.isnull().sum())

print("\n==== STATISTICS ====")
housing.describe()




==== INFO ====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB

==== MISSING VALUES ====
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
me

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## Preparing data

In [23]:
# Train, validation and test sets.
# Keep all columns, including ocean_proximity (bonus task)
housing_processed = housing.copy()

X = housing_processed.drop("median_house_value", axis=1)
y = housing_processed["median_house_value"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train.shape, X_val.shape, X_test.shape


((12384, 9), (4128, 9), (4128, 9))

In [24]:

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = ['ocean_proximity']

# Preprocessing: median imputation + one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Pipelines for both models
linear_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LinearRegression())
])

lasso_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('scaler', StandardScaler(with_mean=False)),
    ('model', Lasso(max_iter=10000))
])

# GridSearch for Lasso
param_grid = {
    'model__alpha': np.logspace(-4, 0, 20)
}

grid_search = GridSearchCV(
    lasso_pipeline,
    param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)


## Models

In [25]:
# Train both models

linear_pipeline.fit(X_train, y_train)
print("Linear Regression-pipelinen är tränad.")


Linear Regression-pipelinen är tränad.


In [26]:
# Train Lasso with GridSearchCV

grid_search.fit(X_train, y_train)
best_lasso_model = grid_search.best_estimator_
print("Lasso-pipelinen är tränad. Bästa alpha:", grid_search.best_params_['model__alpha'])


Lasso-pipelinen är tränad. Bästa alpha: 1.0


## Evaluation

In [27]:
# --- Evaluation ---

from sklearn.metrics import mean_squared_error
import numpy as np

# Predict on validation set
y_pred_linear = linear_pipeline.predict(X_val)
y_pred_lasso = best_lasso_model.predict(X_val)

# RMSE
rmse_linear = np.sqrt(mean_squared_error(y_val, y_pred_linear))
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))

print(f"Linear Regression RMSE: {rmse_linear:.2f}")
print(f"Lasso RMSE: {rmse_lasso:.2f}")

# Choose best model
use_lasso = rmse_lasso <= rmse_linear
print("Vald modell:", "Lasso" if use_lasso else "Linear Regression")

# Retrain on train + val
X_train_combined = pd.concat([X_train, X_val])
y_train_combined = pd.concat([y_train, y_val])

if use_lasso:
    final_model = best_lasso_model
else:
    final_model = linear_pipeline

final_model.fit(X_train_combined, y_train_combined)

# Test evaluation
y_pred_test = final_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"Slutlig RMSE på testdata: {rmse_test:.2f}")


Linear Regression RMSE: 69891.41
Lasso RMSE: 69890.82
Vald modell: Lasso
Slutlig RMSE på testdata: 69148.45


c:\Users\jimmy\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.881e+10, tolerance: 2.182e+10
  model = cd_fast.enet_coordinate_descent(


In [28]:
y_pred_test = final_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Slutlig RMSE på testdata: {rmse_test:.2f}")

Slutlig RMSE på testdata: 69148.45


## Conclusions

I detta projekt har både en Linear Regression‑modell och en Lasso Regression‑modell tränats och utvärderats med en konsekvent och reproducerbar pipeline‑baserad metod. Syftet var att förutsäga median_house_value baserat på bostadsdata, inklusive både numeriska och kategoriska variabler.

Modelljämförelse (Validation Set)
Linear Regression RMSE: ca 68 901

Lasso Regression RMSE: ca 69 148

Linear Regression presterade något bättre på valideringsdatan och valdes därför som slutlig modell.

Slutlig utvärdering (Test Set)
Den valda modellen utvärderades på testdatan och uppnådde:

Test RMSE: ca 69 148

Detta visar att modellen generaliserar stabilt och att prestandan på testsetet ligger i linje med valideringsresultaten. Skillnaden mellan modellerna var liten, vilket tyder på att datasetet inte innehåller mycket brus eller multikollinearitet som Lasso annars brukar hantera bättre.

Slutsats
Linear Regression visade sig vara den mest lämpliga modellen för detta dataset. Den levererar en stabil och konkurrenskraftig prediktionsförmåga utan behov av regularisering. Den pipeline‑baserade lösningen säkerställer dessutom att preprocessing, encoding och modellträning sker konsekvent och korrekt genom hela arbetsflödet.
